# 9장 Multimodal Large Language Models — SageMaker AI Studio(JupyterLab)용

<!-- origin-notice -->
> **원본 노트북:** [chapter09/Chapter 9 - Multimodal Large Language Models.ipynb](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter09/Chapter%209%20-%20Multimodal%20Large%20Language%20Models.ipynb)  
> **원본 저장소:** [HandsOnLLM/Hands-On-Large-Language-Models](https://github.com/HandsOnLLM/Hands-On-Large-Language-Models) (Apache License 2.0, © 2024 Jay Alammar & Maarten Grootendorst)  
> 이 노트북은 원본을 SageMaker AI Studio에서 실행되도록 수정하고 한국어 설명·보충 실험을 추가한 것입니다. 변경 내용은 노트북 안의 "책과 다를 수 있는 점"에 적혀 있습니다.

원본: HandsOnLLM/Hands-On-Large-Language-Models `chapter09` 노트북. 텍스트만 다루던 모델에 **이미지**를 붙이는 두 가지 방법을 실습합니다. 하나는 이미지와 텍스트를 같은 벡터 공간에 넣는 **멀티모달 임베딩**(CLIP)이고, 다른 하나는 이미지를 LLM이 읽을 수 있는 입력으로 바꾸는 **멀티모달 생성 모델**(BLIP-2)입니다.

| 절 | 하는 일 | 책 위치(PDF 쪽) |
|---|---|---|
| 1 | CLIP으로 이미지·캡션을 각각 임베딩하고 유사도 계산 | 366–373 |
| 2 | 이미지 3장 × 캡션 3개 유사도 행렬, sentence-transformers로 같은 일 하기 | 373–376 |
| 3 | (+) 이미지가 몇 개의 **패치 토큰**이 되는지 세기, 주행 장면 **제로샷 분류** | — |
| 4 | BLIP-2 불러오기, 전처리(이미지·텍스트) 살펴보기 | 378–386 |
| 5 | BLIP-2 활용 1: 이미지 캡셔닝 | 387–390 |
| 6 | BLIP-2 활용 2: 이미지에 대해 대화하기(VQA, 채팅형 프롬프트) | 391–394 |
| 7 | (+) 이미지가 LLM 입력 속 **32개 토큰**이 되는 모습, 주행 장면 질의 | — |

`(+ …)` 표시는 책에 없는 보충 실험입니다. 1절 이론(ViT, 패치 임베딩, 361–365쪽)은 코드가 없어 3절 보충 실험에서 숫자로 확인합니다.

**권장 환경.** `ml.g4dn.xlarge`(T4 16GB)로 충분합니다. BLIP-2(약 39억 파라미터)를 fp16으로 올리면 GPU 메모리를 약 8GB 씁니다. 다만 **스토리지가 중요합니다.** BLIP-2 체크포인트는 fp32로 저장되어 있어 내려받는 양이 약 15GB입니다. 앞 장 캐시(Phi-3 약 7.6GB 등)와 합치면 space 스토리지를 **50GB 이상**으로 늘리세요. 체크포인트를 읽는 동안 CPU 메모리(16GB)가 부족해 커널이 죽으면 `ml.g4dn.2xlarge`(32GB)로 바꿔 다시 실행하세요.

## 0. 환경 점검과 패키지 설치

GPU·디스크·버전을 확인하고 없는 패키지만 설치합니다. 이 장은 `transformers`, `sentence-transformers`, `matplotlib`, `scikit-learn`, `ipywidgets`를 씁니다.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h ~ | tail -1
import importlib, subprocess, sys
need = {"sentence_transformers": "sentence-transformers", "sklearn": "scikit-learn",
        "matplotlib": "matplotlib", "ipywidgets": "ipywidgets"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("설치:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("추가 설치 불필요")
import torch, transformers
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "| transformers", transformers.__version__)

이미지를 내려받는 도우미 함수입니다. 책 코드의 `Image.open(urlopen(url))`는 위키미디어 서버가 User-Agent 없는 요청을 거부(HTTP 403)해 6절 로르샤흐 이미지에서 실패합니다. 그래서 User-Agent를 붙이고, 실패하면 회색 이미지로 대신해 다음 셀이 멈추지 않게 했습니다.

In [ ]:
import io
from urllib.request import urlopen, Request
from PIL import Image

def load_image(url, mode="RGB"):
    try:
        req = Request(url, headers={"User-Agent": "hands-on-llm-study/1.0 (educational notebook)"})
        return Image.open(io.BytesIO(urlopen(req, timeout=30).read())).convert(mode)
    except Exception as e:
        print(f"[이미지 다운로드 실패, 회색 이미지로 대체] {url[:80]}… ({type(e).__name__})")
        return Image.new(mode, (512, 512), "gray")

BASE = "https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/chapter09/images/"
puppy_path, cat_path, car_path = BASE + "puppy.png", BASE + "cat.png", BASE + "car.png"

## 1. CLIP: 이미지와 텍스트를 같은 공간에 임베딩하기 (PDF 366–373)

CLIP은 이미지 인코더(ViT)와 텍스트 인코더 두 개로 이루어져 있습니다. (이미지, 캡션) 쌍 4억 개로 **대조 학습**(contrastive learning)을 해서, 짝이 맞는 이미지와 캡션은 가까운 벡터가 되고 짝이 아닌 것은 멀어지게 학습했습니다. 그 결과 이미지 벡터와 텍스트 벡터를 직접 비교할 수 있습니다. 2장의 문장 임베딩을 이미지까지 넓힌 것입니다.

책이 쓰는 이미지는 AI로 생성한 강아지 사진입니다.

In [ ]:
image = load_image(puppy_path)
caption = "a puppy playing in the snow"
image

CLIP은 텍스트용 **토크나이저**, 이미지용 **프로세서**, 그리고 두 인코더를 모두 가진 **모델**로 구성됩니다.

In [ ]:
from transformers import CLIPTokenizerFast, CLIPProcessor, CLIPModel

model_id = "openai/clip-vit-base-patch32"
clip_tokenizer = CLIPTokenizerFast.from_pretrained(model_id)   # 텍스트 전처리
clip_processor = CLIPProcessor.from_pretrained(model_id)       # 이미지 전처리
model = CLIPModel.from_pretrained(model_id)                     # 텍스트·이미지 임베딩 생성

In [ ]:
inputs = clip_tokenizer(caption, return_tensors="pt")
inputs

**관찰 포인트:** `</w>`는 단어의 끝을 표시합니다. 문장 앞뒤에 `<|startoftext|>`, `<|endoftext|>`가 붙습니다(2장에서 본 특수 토큰).

In [ ]:
clip_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

텍스트 임베딩을 만듭니다.

> **책과 다를 수 있는 점.** transformers 4.x에서는 `get_text_features()`가 512차원 텐서를 바로 돌려줍니다. 5.x부터는 출력 객체를 돌려주고, 같은 값이 `.pooler_output`에 들어 있습니다(직접 비교해 동일함을 확인). 아래 `features()` 함수는 두 버전 모두에서 텐서를 꺼냅니다.

In [ ]:
def features(x):
    return x.pooler_output if hasattr(x, "pooler_output") else x

text_embedding = features(model.get_text_features(**inputs))
text_embedding.shape

이미지는 프로세서가 **224×224**로 자르고 정규화합니다. 결과 모양은 `[배치, RGB 3채널, 높이, 너비]`입니다.

In [ ]:
processed_image = clip_processor(text=None, images=image, return_tensors="pt")["pixel_values"]
processed_image.shape

전처리된 이미지를 그려 봅니다. 정규화 때문에 색이 이상하게 보이는 것이 정상입니다(값이 0~1 범위를 벗어나 잘린다는 경고가 나올 수 있습니다).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

img = processed_image.squeeze(0)
img = img.permute(*torch.arange(img.ndim - 1, -1, -1))
img = np.einsum("ijk->jik", img.numpy())
plt.imshow(img); plt.axis("off")

In [ ]:
image_embedding = features(model.get_image_features(processed_image))
image_embedding.shape

두 벡터를 길이 1로 정규화한 뒤 내적하면 코사인 유사도입니다. **관찰 포인트:** CLIP의 이미지-텍스트 유사도는 0.3 안팎이면 잘 맞는 편입니다. 2장 문장 임베딩처럼 0.8~0.9가 나오지 않습니다. 두 모달리티 벡터 사이에 일정한 간격(modality gap)이 있기 때문이고, 그래서 **절댓값보다 후보끼리의 상대 순위**로 판단합니다.

In [ ]:
text_embedding /= text_embedding.norm(dim=-1, keepdim=True)
image_embedding /= image_embedding.norm(dim=-1, keepdim=True)

text_embedding = text_embedding.detach().cpu().numpy()
image_embedding = image_embedding.detach().cpu().numpy()
score = text_embedding @ image_embedding.T
score

## 2. 이미지 3장 × 캡션 3개 유사도 행렬 (PDF 373–376)

강아지·고양이·슈퍼카 이미지와 각각의 캡션을 모두 임베딩해 교차 비교합니다. 대각선(짝이 맞는 쌍)이 가장 밝아야 합니다.

In [ ]:
paths = [puppy_path, cat_path, car_path]
images = [load_image(p, "RGBA") for p in paths]
captions = [
    "a puppy playing in the snow",
    "a pixelated image of a cute cat",
    "A supercar on the road \nwith the sunset in the background",
]

image_embeddings = []
for img_ in images:
    image_processed = clip_processor(images=img_, return_tensors="pt")["pixel_values"]
    image_embeddings.append(features(model.get_image_features(image_processed)).detach().cpu().numpy()[0])
image_embeddings = np.array(image_embeddings)

text_embeddings = []
for c in captions:
    inputs = clip_tokenizer(c, return_tensors="pt")
    text_embeddings.append(features(model.get_text_features(**inputs)).detach().cpu().numpy()[0])
text_embeddings = np.array(text_embeddings)

from sklearn.metrics.pairwise import cosine_similarity
sim_matrix = cosine_similarity(image_embeddings, text_embeddings)

In [ ]:
plt.figure(figsize=(20, 14))
plt.imshow(sim_matrix, cmap="viridis")
plt.yticks(range(len(captions)), captions, fontsize=18)
plt.xticks([])
for i, img_ in enumerate(images):
    plt.imshow(img_, extent=(i - 0.5, i + 0.5, -1.6, -0.6), origin="lower")
for x in range(sim_matrix.shape[1]):
    for y in range(sim_matrix.shape[0]):
        plt.text(x, y, f"{sim_matrix[y, x]:.2f}", ha="center", va="center", size=30)
for side in ["left", "top", "right", "bottom"]:
    plt.gca().spines[side].set_visible(False)
plt.xlim([-0.5, len(captions) - 0.5]); plt.ylim([len(captions) + 0.5, -2])
plt.savefig("sim_matrix.png", dpi=150, bbox_inches="tight")

`sentence-transformers`에도 CLIP이 들어 있어, 2장에서 쓴 `encode()` 한 줄로 이미지와 텍스트를 모두 임베딩할 수 있습니다. 행이 이미지, 열이 캡션입니다.

In [ ]:
from sentence_transformers import SentenceTransformer, util

st_clip = SentenceTransformer("clip-ViT-B-32")
image_embeddings = st_clip.encode(images)
text_embeddings = st_clip.encode(captions)
sim_matrix = util.cos_sim(image_embeddings, text_embeddings)
print(sim_matrix)

## 3. (+ 보충) 이미지는 몇 개의 토큰이 될까, 주행 장면을 말로 분류하기

### 3-1. 패치 토큰 세기

ViT(책 361–365쪽)는 이미지를 정사각형 **패치**로 잘라 각 패치를 단어 토큰처럼 다룹니다. `clip-vit-base-patch32`는 224×224 이미지를 32×32 패치로 자르므로 (224/32)² = **49개 패치 + 분류용 [CLS] 1개 = 50개 토큰**이 됩니다. 비전 인코더의 출력 모양으로 확인합니다.

In [ ]:
with torch.no_grad():
    vision_out = model.vision_model(pixel_values=processed_image)
print("비전 인코더 출력:", tuple(vision_out.last_hidden_state.shape), "= [배치, 토큰 수, 차원]")

패치를 작게 하거나 해상도를 높이면 토큰 수가 제곱으로 늘어납니다. 설정 파일(config)만 받아 계산하므로 가중치 다운로드는 없습니다.

In [ ]:
import pandas as pd
from transformers import CLIPVisionConfig
rows = []
for m in ["openai/clip-vit-base-patch32", "openai/clip-vit-base-patch16", "openai/clip-vit-large-patch14-336"]:
    c = CLIPVisionConfig.from_pretrained(m)
    n = (c.image_size // c.patch_size) ** 2
    rows.append({"model": m, "image_size": c.image_size, "patch": c.patch_size, "patches": n, "tokens(+CLS)": n + 1,
                 "6 cameras": 6 * (n + 1)})
pd.DataFrame(rows).set_index("model")

**읽는 법.** 마지막 열은 카메라 6대 영상을 프레임 하나씩만 넣어도 생기는 토큰 수입니다. 1장 Q&A에서 다룬 셀프 어텐션의 n² 비용이 자율주행 모델에서 왜 문제가 되는지, 그리고 BLIP-2(7절)처럼 이미지를 **소수의 토큰으로 압축**하는 장치가 왜 필요한지가 여기서 드러납니다.

### 3-2. 주행 장면 제로샷 분류

CLIP은 분류용으로 따로 학습하지 않아도, 후보 문장들과의 유사도를 비교해 이미지를 분류할 수 있습니다(제로샷, zero-shot). 횡단보도 앞에 차들이 멈춰 선 실제 사진에 주행 상황 문장 6개를 붙여 봅니다.

이미지 출처: [Wikimedia Commons, "Cars stopped at Bell Street signalised pedestrian crossing, August 2024"](https://commons.wikimedia.org/wiki/File:Cars_stopped_at_Bell_Street_signalised_pedestrian_crossing,_August_2024.jpg), 촬영 PEPSI697, CC BY-SA 4.0.

In [ ]:
road_url = ("https://upload.wikimedia.org/wikipedia/commons/thumb/7/70/"
            "Cars_stopped_at_Bell_Street_signalised_pedestrian_crossing%2C_August_2024.jpg/"
            "960px-Cars_stopped_at_Bell_Street_signalised_pedestrian_crossing%2C_August_2024.jpg")
road = load_image(road_url)
road.resize((480, 360))

In [ ]:
labels = ["a pedestrian crossing the street in front of a car",
          "cars stopped at a red traffic light",
          "an empty highway",
          "a parking lot",
          "a traffic jam on a highway",
          "a snowy mountain road"]
inp = clip_processor(text=["a photo of " + l for l in labels], images=road, return_tensors="pt", padding=True)
with torch.no_grad():
    probs = model(**inp).logits_per_image.softmax(dim=-1)[0]
pd.Series(probs.numpy(), index=labels).sort_values(ascending=False).round(3)

**관찰 포인트.** 직접 실행했을 때 "cars stopped at a red traffic light"가 약 0.9로 1위였습니다. 후보 문장을 바꾸면 결과가 크게 달라지는 점도 확인해 보세요. 예를 들어 "a pedestrian is about to step into the road"를 넣거나, "a photo of" 접두어를 빼 보면 됩니다. CLIP은 **주어진 후보 중에서만** 고르기 때문에, 후보에 없는 위험 상황은 알아채지 못합니다. 이것이 주행 장면 검색·라벨링에 CLIP을 쓸 때의 기본 한계입니다.

In [ ]:
import gc
del model, st_clip, vision_out
gc.collect(); torch.cuda.empty_cache()

## 4. BLIP-2: 이미지를 LLM이 읽는 입력으로 바꾸기 (PDF 378–386)

BLIP-2는 이미 학습된 **비전 인코더(ViT)**와 **LLM(OPT-2.7B)**을 둘 다 얼려 두고(frozen), 그 사이에 **Q-Former**라는 작은 다리만 학습합니다. Q-Former는 이미지 특징에서 학습 가능한 쿼리 32개로 필요한 정보를 뽑아 LLM의 임베딩 공간으로 투영합니다. LLM 입장에서는 이 32개 벡터가 문장 앞에 붙은 "소프트 시각 프롬프트(soft visual prompt)"입니다.

> **책과 다를 수 있는 점.** 공식 노트북은 `revision="51572668…"`로 옛 버전을 고정합니다. 그 버전의 프로세서는 이미지 토큰 정보(`num_query_tokens`)가 없는 옛 방식입니다. 최신 transformers는 프로세서가 텍스트 앞에 `<image>` 토큰 32개를 직접 끼워 넣는 새 방식을 씁니다. 그래서 이 노트북은 **고정하지 않고 최신 버전**을 씁니다(`BLIP_REVISION = None`). 책이 고정한 transformers 4.41 환경을 그대로 쓴다면 고정 revision을 넣으세요.

처음 실행하면 약 15GB를 내려받습니다.

In [ ]:
from transformers import AutoProcessor, Blip2ForConditionalGeneration

BLIP_ID = "Salesforce/blip2-opt-2.7b"
BLIP_REVISION = None   # transformers 4.46 이하라면 "51572668da0eb669e01a189dc22abe6088589a24"

blip_processor = AutoProcessor.from_pretrained(BLIP_ID, revision=BLIP_REVISION)
model = Blip2ForConditionalGeneration.from_pretrained(
    BLIP_ID, revision=BLIP_REVISION,
    torch_dtype=torch.float16,          # transformers 5.x에서는 dtype= (기존 이름은 경고만 나고 동작)
)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"GPU 사용량: {torch.cuda.memory_allocated()/1e9:.1f} GB")

책의 TIP처럼 세 부품을 확인합니다. 비전 인코더와 LLM은 이름만 봐도 따로 학습된 기존 모델이고, `qformer`가 그 둘을 잇는 다리입니다.

In [ ]:
print("vision :", type(model.vision_model).__name__)
print("bridge :", type(model.qformer).__name__, "| query tokens:", model.config.num_query_tokens)
print("LLM    :", type(model.language_model).__name__)

### 4-1. 이미지 전처리

슈퍼카 이미지가 224×224 정사각형으로 바뀝니다. 책은 이 이미지를 "520 × 492의 아주 넓은 이미지"라고 설명하지만, 실제 파일은 가로 492 × 세로 520으로 거의 정사각형입니다(아래 출력의 `원본 크기`는 PIL 관례대로 (가로, 세로)). **관찰 포인트:** 가로세로 비율이 크게 다른 이미지는 정사각형으로 찌그러집니다. 주행 카메라는 대부분 16:9처럼 가로로 긴 영상이라 이 점이 실제로 문제가 됩니다.

In [ ]:
image = load_image(car_path)
print("원본 크기:", image.size)
inputs = blip_processor(image, return_tensors="pt").to(device, torch.float16)
inputs["pixel_values"].shape

### 4-2. 텍스트 전처리

BLIP-2의 LLM(OPT)은 GPT-2 계열 BPE 토크나이저를 씁니다.

In [ ]:
blip_processor.tokenizer

`Ġ`는 공백을 뜻합니다. 바이트 단위 BPE가 공백(코드 포인트 32)을 출력 가능한 문자로 옮기려고 256을 더해 `Ġ`(288)로 표시한 것입니다.

> **책과 다를 수 있는 점.** 최신 프로세서는 `input_ids` 앞에 `<image>` 토큰 32개를 붙이므로, 책처럼 전체를 출력하면 `<image>`가 길게 이어집니다. 아래 셀은 텍스트 부분만 보도록 `<image>`를 걸러 냅니다(7절에서 이 32개를 따로 봅니다).

In [ ]:
text = "Her vocalization was remarkably melodic"
token_ids = blip_processor(image, text=text, return_tensors="pt")["input_ids"][0]
tokens = blip_processor.tokenizer.convert_ids_to_tokens(token_ids)
tokens = [t for t in tokens if t != "<image>"]
print(tokens)
print([t.replace("Ġ", "_") for t in tokens])

## 5. 활용 1: 이미지 캡셔닝 (PDF 387–390)

이미지만 넣고 생성하면 BLIP-2는 캡션을 씁니다. 책 출력은 "an orange supercar driving on the road at sunset"입니다.

In [ ]:
def caption(img, max_new_tokens=20):
    inputs = blip_processor(img, return_tensors="pt").to(device, torch.float16)
    generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return blip_processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

image = load_image(car_path)
caption(image)

책의 재미있는 예시: 로르샤흐 잉크 얼룩 그림(퍼블릭 도메인). 책 출력은 "a black and white ink drawing of a bat"입니다.

In [ ]:
url = "https://upload.wikimedia.org/wikipedia/commons/7/70/Rorschach_blot_01.jpg"
rorschach = load_image(url)
display(rorschach.resize((300, 200)))
caption(rorschach)

## 6. 활용 2: 이미지에 대해 대화하기 (PDF 391–394)

이미지와 함께 `Question: … Answer:` 형식의 프롬프트를 넣으면 시각 질의응답(VQA, visual question answering)이 됩니다. 이전 질문과 답을 프롬프트에 이어 붙이면 채팅처럼 쓸 수 있습니다. 모델이 대화를 "기억"하는 것이 아니라, **우리가 매번 대화 기록을 입력에 다시 넣어 주는 것**입니다(7장 Memory와 같은 원리).

In [ ]:
def ask(img, prompt, max_new_tokens=30):
    inputs = blip_processor(img, text=prompt, return_tensors="pt").to(device, torch.float16)
    generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return blip_processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

image = load_image(car_path)
ask(image, "Question: Write down what you see in this picture. Answer:")

이전 답을 프롬프트에 넣고 후속 질문을 합니다. 책 출력은 "$1,000,000"입니다. 그럴듯하지만 근거 없는 숫자라는 점도 눈여겨보세요.

In [ ]:
ask(image, "Question: Write down what you see in this picture. Answer: A sports car driving on the road at sunset. "
           "Question: What would it cost me to drive that car? Answer:")

`ipywidgets`로 만든 채팅 상자입니다. 입력란에 질문을 쓰고 Enter를 누르면 답이 위에 쌓입니다. 위젯이 표시되지 않으면 바로 위의 `ask()` 함수로 대신하세요.

In [ ]:
from IPython.display import HTML, display
import ipywidgets as widgets

def text_eventhandler(*args):
    question = args[0]["new"]
    if question:
        args[0]["owner"].value = ""
        if not memory:
            prompt = " Question: " + question + " Answer:"
        else:
            template = "Question: {} Answer: {}."
            prompt = " ".join(template.format(q, a) for q, a in memory) + " Question: " + question + " Answer:"
        inputs = blip_processor(image, text=prompt, return_tensors="pt").to(device, torch.float16)
        generated_ids = model.generate(**inputs, max_new_tokens=100)
        generated_text = blip_processor.batch_decode(generated_ids, skip_special_tokens=True)
        generated_text = generated_text[0].strip().split("Question")[0]
        memory.append((question, generated_text))
        output.append_display_data(HTML("<b>USER:</b> " + question))
        output.append_display_data(HTML("<b>BLIP-2:</b> " + generated_text))
        output.append_display_data(HTML("<br>"))

in_text = widgets.Text()
in_text.continuous_update = False
in_text.observe(text_eventhandler, "value")
output = widgets.Output()
memory = []
display(widgets.VBox(children=[output, in_text],
                     layout=widgets.Layout(display="inline-flex", flex_flow="column-reverse")))

## 7. (+ 보충) 이미지는 LLM 입력 속 32개 토큰이 된다

최신 프로세서가 만든 `input_ids`를 세어 보면 이미지 자리(`<image>`)가 정확히 `num_query_tokens`(32)개입니다. LLM은 이 자리에 Q-Former 출력 32개를 받아 텍스트 토큰과 똑같이 셀프 어텐션으로 처리합니다. BLIP-2의 비전 인코더(ViT-g, 224×224 이미지를 14×14 패치로 자름)가 만든 패치 토큰 257개를 32개로 줄여 넘기는 셈입니다.

In [ ]:
ids = blip_processor(road, text="Question: Is it safe to cross? Answer:", return_tensors="pt")["input_ids"][0]
toks = blip_processor.tokenizer.convert_ids_to_tokens(ids)
n_img = sum(t == "<image>" for t in toks)
print(f"전체 {len(toks)}토큰 = 이미지 {n_img} + 텍스트 {len(toks) - n_img}")
with torch.no_grad():
    vt = model.vision_model(pixel_values=blip_processor(road, return_tensors="pt")["pixel_values"].to(device, torch.float16))
print("ViT 패치 토큰:", vt.last_hidden_state.shape[1], "→ Q-Former 출력:", model.config.num_query_tokens)

3-2의 주행 장면 사진에 질문해 봅니다. **관찰 포인트:** BLIP-2는 2023년 모델이고 주행 데이터로 학습하지 않았습니다. 답이 그럴듯해도 신호 색, 보행자 유무 같은 안전 판단을 근거 있게 하지 못합니다. 같은 질문을 표현만 바꿔 여러 번 해 보고 답이 흔들리는지 확인하세요.

In [ ]:
for q in ["What do you see in this picture?",
          "What color is the traffic light?",
          "Is there a pedestrian on the crossing?",
          "Should the cars wait or go?"]:
    print(f"Q: {q}\nA: {ask(road, 'Question: ' + q + ' Answer:')}\n")

In [ ]:
del model, vt; gc.collect(); torch.cuda.empty_cache()
print(f"GPU 사용량: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 8. 정리

| 방식 | 대표 모델 | 입력 → 출력 | 이미지의 표현 | 이 노트북 |
|---|---|---|---|---|
| 멀티모달 임베딩 | CLIP | 이미지 또는 텍스트 → 벡터 1개 | 패치 토큰 50개 → 벡터 1개 | 1–3절 |
| 멀티모달 생성 | BLIP-2 | 이미지 + 텍스트 → 텍스트 | 패치 토큰 257개 → Q-Former 32개 → LLM 입력 | 4–7절 |

**자율주행·Agent로 이어지는 점(책 밖).** 최근 주행용 VLM·VLA도 BLIP-2와 같은 뼈대를 씁니다. 사전학습된 비전 인코더가 있고, 그 출력을 LLM 임베딩 공간으로 옮기는 연결부(Q-Former, MLP 프로젝터 등)가 있으며, 그 뒤에 LLM이 있습니다. 차이는 세 가지입니다. 입력이 카메라 여러 대·여러 프레임이고, 출력이 텍스트 대신 또는 텍스트와 함께 **궤적·행동 토큰**이며, 주행 데이터로 추가 학습합니다. 3-1에서 본 토큰 수 문제 때문에 연결부의 토큰 압축 방식이 핵심 설계 항목입니다. CLIP 계열 임베딩은 대규모 주행 로그에서 "보행자가 무단횡단하는 장면"처럼 **문장으로 장면을 검색**하는 데 많이 쓰입니다(8장 시맨틱 검색과 같은 원리).

**끝나면** Studio에서 space를 **Stop** 하세요. BLIP-2 캐시(약 15GB)가 더 필요 없으면 `~/.cache/huggingface/hub/models--Salesforce--blip2-opt-2.7b`를 지워 스토리지를 확보할 수 있습니다.